# Read Database Module

This notebook implements the `read_database` function to interact with the boxers SQLite database. It allows filtering and searching boxers dynamically.

In [ ]:
import sqlite3
from typing import Optional, Dict, Any

def read_database(id: Optional[int] = None, name: Optional[str] = None) -> Dict[int, Dict[str, Any]]:
    """
    Retrieve boxer records from the SQLite database with optional filters.

    Queries the 'boxers' table. Filters by exact ID if supplied, or searches
    by partial name (case-insensitive) if specified. If no filters are provided,
    it retrieves all boxer records.

    Args:
        id (Optional[int]): The unique identifier of the boxer. Defaults to None.
        name (Optional[str]): Partial name of the boxer to search for. Defaults to None.

    Returns:
        Dict[int, Dict[str, Any]]: A dictionary mapped by boxer ID containing boxer details.
    """
    # Connect to the project's database
    connection = sqlite3.connect("boxers.db")
    cursor = connection.cursor()

    # Dynamically build the SQL query and its parameters
    query = "SELECT id, name, total_fights, wins, loses, draws FROM boxers"
    params = []
    conditions = []

    if id is not None:
        conditions.append("id = ?")
        params.append(id)
    if name is not None:
        conditions.append("name LIKE ?")
        params.append(f"%{name}%")

    if conditions:
        query += " WHERE " + " AND ".join(conditions)

    cursor.execute(query, params)
    rows = cursor.fetchall()
    connection.close()

    # Map results into a dictionary matching project specifications
    boxers_dict = {}
    for row in rows:
        boxer_id = row[0]
        boxers_dict[boxer_id] = {
            "name": row[1],
            "total_fights": row[2],
            "wins": row[3],
            "loses": row[4],
            "draws": row[5]
        }

    # Print formatted output to the terminal in English
    if not boxers_dict:
        print("No boxers found matching the search criteria.")
    else:
        print(f"{'ID':<5} | {'Name':<25} | {'Total':<6} | {'Wins':<9} | {'Loses':<9} | {'Draws':<7}")
        print("-" * 75)
        for b_id, data in boxers_dict.items():
            print(f"{b_id:<5} | {data['name']:<25} | {data['total_fights']:<6} | {data['wins']:<9} | {data['loses']:<9} | {data['draws']:<7}")

    return boxers_dict
